# Métricas de por tarea, mirando lo que queda en el subgrafo


- factoid:
  - medidas: estricta (el primer candidato es oro), laxa (algún candidato entre los 5 primeros es oro), MRR (pondera por posicion)
  - contra el oro: sinónimos de BioASQ o el nodo del oro, por el diccionario de linkear_entidades
- list:
  - medidas: precisión, recall y F1 por pregunta, comparando la lista con los ítems del oro como conjuntos
  - contra el oro: idem
- yes/no:
  - medidas: exactitud y F1 macro sobre las dos clases
  - contra el oro: la primera palabra en minúsculas
- summary:
  - medidas: cobertura y contradicción con el fact checker (RAGChecker, rc_\*), y ROUGE-2 y ROUGE-SU4 (F1 y recall), las medidas automáticas de BioASQ
  - contra el oro: rc_\*: la primera ideal, la del experto; ROUGE: la mejor de las ideales. No hay oro de entidades

Para cada tarea se mide la media de las 3 repeticiones y la diferencia con grafo_entero (el GraphRAG estático) y contra sin_grafo (el modelo solo)



In [ ]:
import json
import os
from collections import Counter
import re
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, "../../graphrag")
from entidades.linkear_entidades import Linkeador

TAREAS = ["factoid", "list", "yesno", "summary"]
CONFIGURACIONES = ["sin_grafo", "grafo_entero", "semillas", "diamond", "diamond_c3", "completa"]
SELECCION = ["semillas", "diamond", "diamond_c3", "completa"]
# grupos de cobertura del oro para list: hasta la mitad de los items en el subgrafo (fraccion <= 0.5) o mas de la mitad
GRUPOS = ["hasta_la_mitad", "mas_de_la_mitad"]
REFERENCIAS = ["grafo_entero", "sin_grafo"]
MEDIDA_PRINCIPAL = {"factoid": "laxa", "list": "f1", "yesno": "acierto", "summary": "rc_cobertura"}
REPETICIONES = 3
#la frase fija de abstencion de la busqueda global
ABSTENCION = "I am sorry but I am unable to answer this question given the provided data"
#la marca con la que el reduce senala lo que no sale de los reportes
MARCA = re.compile(r"\s*[\[(]\s*LLM:?\s*verify\s*[\])]", re.IGNORECASE)
nodos_de_item = {}

pd.set_option("display.width", 250)

linkeador = Linkeador()

## funciones

In [ ]:
def cargar(tarea):
    "la muestra, las claves de los conjuntos de entidades y las respuestas con codigo 0 que haya en disco"
    with open(f"preguntas/muestra_{tarea}.json") as archivo:
        muestra = json.load(archivo)
    ruta_claves = os.path.join(tarea, "claves.json")
    claves = json.load(open(ruta_claves)) if os.path.exists(ruta_claves) else {}
    filas = []
    for rep in range(1, REPETICIONES + 1):
        for nombre in muestra:
            for configuracion in CONFIGURACIONES:
                ruta = os.path.join(tarea, "respuestas", f"rep{rep}", nombre, configuracion, "respuesta.json")
                if not os.path.exists(ruta):
                    continue
                with open(ruta) as archivo:
                    respuesta = json.load(archivo)
                if respuesta["codigo"] != 0:
                    continue
                filas.append({"consulta": nombre, "rep": rep, "configuracion": configuracion,
                              "respuesta": respuesta["respuesta"], "abstiene": ABSTENCION in respuesta["respuesta"],
                              "segundos": respuesta.get("segundos")})
    return muestra, claves, pd.DataFrame(filas)


def normalizar(texto):
    "un item en la forma en que se compara: sin la marca, citas, parentesis, vinetas ni numeracion y en minusculas"
    texto = MARCA.sub(" ", texto)
    texto = re.sub(r"\[Data:[^\]]*\]", " ", texto)
    texto = re.sub(r"\([^)]*\)", " ", texto)
    texto = re.sub(r"^\s*([-*•]|\d+[.)])\s*", "", texto)
    return texto.strip(" \t*.,;:").lower()


def lista_de(respuesta):
    """los items del primer bloque de lineas, normalizados, sin repetir y sin la palabra None. una primera linea que
    termina en ":" es un encabezado y la lista empieza despues"""
    lineas = respuesta.strip().splitlines()
    if lineas and lineas[0].strip().endswith(":"):
        lineas = "\n".join(lineas[1:]).strip().splitlines()
    items = []
    for linea in lineas:
        if not linea.strip():
            break
        if MARCA.search(linea) and not MARCA.sub("", linea).strip():
            continue
        item = normalizar(linea)
        if item and item != "none" and item not in items:
            items.append(item)
    return items



def nodos_de(item):
    # los nodos a los que el diccionario lleva un item de la respuesta, en los tres tipos (como el oro en armar_conjuntos)
    if item not in nodos_de_item:
        nodos_de_item[item] = {(tipo, id_nodo) for tipo in ["Gene", "Disease", "Drug"] for id_nodo in linkeador.exacto(tipo, [item])}
    return nodos_de_item[item]


def acierta(item, sinonimos_normalizados, nodos_del_item):
    "1 si esta contenido item en sinonimos_normalizados, 0 cc"
    return item in sinonimos_normalizados or bool(nodos_de(item) & nodos_del_item)


def medir_factoid(respuestas, muestra):
    "para cada respuesta de la muestra para factoid se calculan las tres metricas (estricta, laxa y mrr)"
    filas = []
    for respuesta in respuestas.itertuples():
        consulta = muestra[respuesta.consulta]
        sinonimos = {normalizar(s) for item in consulta["oro"] for s in item}
        nodos_del_oro = {tuple(nodo) for nodos in consulta["oro_nodos"] for nodo in nodos}
        candidatos = [] if respuesta.abstiene else lista_de(respuesta.respuesta)[:5]
        posicion = next((puesto for puesto, c in enumerate(candidatos, start=1) if acierta(c, sinonimos, nodos_del_oro)), None)
        filas.append({"estricta": int(posicion == 1), "laxa": int(posicion is not None), "mrr": 1 / posicion if posicion else 0,
                      "laxa_sinonimos": int(any(c in sinonimos for c in candidatos)), "candidatos": " | ".join(candidatos)})
    return pd.DataFrame(filas)


def medir_list(respuestas, muestra):
    "para cada respuesta de la muestra calcula el pricision, recall y f1. Tambien se normaliza, como en medir_factoid"
    filas = []
    for respuesta in respuestas.itertuples():
        consulta = muestra[respuesta.consulta]
        items = [] if respuesta.abstiene else lista_de(respuesta.respuesta)
        items_acertados, oro_acertado = set(), 0
        acertados_sinonimos, oro_sinonimos, solo_diccionario = set(), 0, []
        for sinonimos, nodos in zip(consulta["oro"], consulta["oro_nodos"]):
            normalizados = {normalizar(s) for s in sinonimos}
            nodos_del_item = {tuple(nodo) for nodo in nodos}
            coinciden = [item for item in items if acierta(item, normalizados, nodos_del_item)]
            if coinciden:
                oro_acertado += 1
                items_acertados.update(coinciden)
            por_sinonimo = [item for item in coinciden if item in normalizados]
            if por_sinonimo:
                oro_sinonimos += 1
                acertados_sinonimos.update(por_sinonimo)
            else:
                solo_diccionario += [f"{item} = {sinonimos[0]}" for item in coinciden]
        precision = len(items_acertados) / len(items) if items else 0
        recall = oro_acertado / len(consulta["oro"])
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
        precision_s = len(acertados_sinonimos) / len(items) if items else 0
        recall_s = oro_sinonimos / len(consulta["oro"])
        f1_sinonimos = 2 * precision_s * recall_s / (precision_s + recall_s) if precision_s + recall_s else 0
        filas.append({"precision": precision, "recall": recall, "f1": f1, "f1_sinonimos": f1_sinonimos, "items": len(items),
                      "lista": " | ".join(items), "solo_diccionario": " | ".join(solo_diccionario)})
    return pd.DataFrame(filas)


def medir_yesno(respuestas, muestra):
    """para cada respuesta de la muestra se anota cual era la respuesta correcta (yes/no), la respuesta dada 
    y si hubo un acierto del sistema"""
    filas = []
    for respuesta in respuestas.itertuples():
        palabras = re.findall(r"[a-z]+", respuesta.respuesta.lower())
        dada = palabras[0] if palabras and palabras[0] in ("yes", "no") and not respuesta.abstiene else None
        oro = muestra[respuesta.consulta]["oro"]
        filas.append({"oro": oro, "dada": dada, "acierto": int(dada == oro)})
    return pd.DataFrame(filas)


def f1_macro(datos):
    "la media de la F1 de la clase yes y la de la clase no"
    f1_por_clase = []
    for clase in ["yes", "no"]:
        verdaderos = ((datos["dada"] == clase) & (datos["oro"] == clase)).sum()
        dadas, reales = (datos["dada"] == clase).sum(), (datos["oro"] == clase).sum()
        precision = verdaderos / dadas if dadas else 0
        recall = verdaderos / reales if reales else 0
        f1_por_clase.append(2 * precision * recall / (precision + recall) if precision + recall else 0)
    return float(np.mean(f1_por_clase))


def palabras_de(texto):
    "las palabras de un texto en minusculas, sin las citas de graphrag ni la marca"
    texto = MARCA.sub(" ", texto)
    texto = re.sub(r"\[Data:[^\]]*\]", " ", texto)
    return re.findall(r"[a-z0-9]+", texto.lower())


def pares_de(palabras, salto):
    "los pares de palabras en orden con hasta salto palabras entre las dos, contados. con salto 0 son los bigramas"
    return Counter((palabras[i], palabras[j]) for i in range(len(palabras)) for j in range(i + 1, min(i + 2 + salto, len(palabras))))


def rouge(respuesta, referencia):
    "recall y F1 de los pares (o palabras) de la respuesta contra los de la referencia"
    comunes = sum((respuesta & referencia).values())
    precision = comunes / sum(respuesta.values()) if respuesta else 0
    recall = comunes / sum(referencia.values()) if referencia else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    return recall, f1


#el fact checker (medir_summary_ragchecker.py): las afirmaciones de la
#primera ideal, extraidas una vez por pregunta, contra la respuesta entera, con gpt-oss:20b de juez
COLUMNAS_RC = ["rc_cobertura", "rc_contradichas", "rc_con_contradiccion", "rc_afirmaciones_ideal"]
RC_SIN_MEDIR = dict.fromkeys(COLUMNAS_RC, np.nan)


def medidas_rc():
    """por respuesta (rep, consulta, configuracion), lo que deja medir_summary_ragchecker.py en summary/ragchecker.json: la
    cobertura (el recall de RAGChecker, afirmaciones de la ideal que la respuesta implica; lo que agrega no resta), las
    afirmaciones de la ideal que la respuesta contradice (la etiqueta Contradiction de response2answer), si hay alguna, y
    cuantas afirmaciones tiene la ideal (el denominador: response2answer tiene una etiqueta por afirmacion). Queda vacio si
    todavia no se corrio"""
    ruta = os.path.join("summary", "ragchecker.json")
    if not os.path.exists(ruta):
        return {}
    with open(ruta) as archivo:
        resultados = json.load(archivo)["results"]
    medidas = {}
    for resultado in resultados:
        rep, consulta, configuracion = resultado["query_id"].split("/")
        contradichas = resultado["response2answer"].count("Contradiction")
        medidas[(int(rep[3:]), consulta, configuracion)] = {
            "rc_cobertura": resultado["metrics"]["recall"], "rc_contradichas": contradichas,
            "rc_con_contradiccion": int(contradichas > 0), "rc_afirmaciones_ideal": len(resultado["response2answer"])}
    return medidas


def medidas_bertscore():
    "por respuesta (rep, consulta, configuracion), el BERTScore contra la mejor ideal que deja medir_bertscore.py"
    tabla = pd.read_csv(os.path.join("summary", "bertscore.csv"))
    return tabla.set_index(["rep", "consulta", "configuracion"])["bertscore"].to_dict()


def medir_summary(respuestas, muestra):
    """ROUGE-2 y ROUGE-SU4 de cada respuesta contra la mejor de las respuestas ideales de su pregunta. ROUGE-2 cuenta los
    bigramas en comun; ROUGE-SU4 los pares con hasta 4 palabras de salto mas las palabras sueltas. rouge_su4_primera es
    contra la primera ideal, la que usa el fact checker. mas BERTScore y las medidas del fact checker (medidas_rc). 
    una abstencion vale cero en todas"""
    rc = medidas_rc()
    bertscore = medidas_bertscore()
    filas = []
    for respuesta in respuestas.itertuples():
        palabras = [] if respuesta.abstiene else palabras_de(respuesta.respuesta)
        bigramas, su4 = pares_de(palabras, 0), pares_de(palabras, 4) + Counter((p,) for p in palabras)
        fila = {"rouge_2": 0, "rouge_2_recall": 0, "rouge_su4": 0, "rouge_su4_recall": 0, "palabras": len(palabras)}
        for ideal in muestra[respuesta.consulta]["ideal_answer"]:
            de_la_ideal = palabras_de(ideal)
            recall_2, f1_2 = rouge(bigramas, pares_de(de_la_ideal, 0))
            recall_su4, f1_su4 = rouge(su4, pares_de(de_la_ideal, 4) + Counter((p,) for p in de_la_ideal))
            for medida, valor in [("rouge_2", f1_2), ("rouge_2_recall", recall_2), ("rouge_su4", f1_su4), ("rouge_su4_recall", recall_su4)]:
                fila[medida] = max(fila[medida], valor)
        primera = palabras_de(muestra[respuesta.consulta]["ideal_answer"][0])
        fila["rouge_su4_primera"] = rouge(su4, pares_de(primera, 4) + Counter((p,) for p in primera))[1]
        fila["bertscore"] = 0 if respuesta.abstiene else bertscore[(respuesta.rep, respuesta.consulta, respuesta.configuracion)]
        fila.update(rc.get((respuesta.rep, respuesta.consulta, respuesta.configuracion), RC_SIN_MEDIR) if not respuesta.abstiene
                    else dict.fromkeys(COLUMNAS_RC, 0))
        filas.append(fila)
    return pd.DataFrame(filas)


MEDIR = {"factoid": medir_factoid, "list": medir_list, "yesno": medir_yesno, "summary": medir_summary}
COLUMNAS = {"factoid": ["estricta", "laxa", "mrr", "laxa_sinonimos"], "list": ["precision", "recall", "f1", "f1_sinonimos"], "yesno": ["acierto"],
            "summary": ["rouge_2", "rouge_2_recall", "rouge_su4", "rouge_su4_recall", "rouge_su4_primera", "bertscore"] + COLUMNAS_RC}


def comparar(medidas, columnas):
    """para cada pregunta se promedian las repeticiones, solo entran las preguntas que tienen las seis configuraciones. la media de
    cada configuracion y la diferencia con el estatico"""
    por_pregunta = medidas.groupby(["configuracion", "consulta"])[columnas].mean()
    completas = medidas.groupby("consulta")["configuracion"].nunique()
    consultas = sorted(completas[completas == len(CONFIGURACIONES)].index)
    if not consultas:
        return None, consultas
    filas = []
    for medida in columnas:
        for referencia in REFERENCIAS:
            de_referencia = por_pregunta.loc[referencia, medida].reindex(consultas).to_numpy()
            for configuracion in CONFIGURACIONES:
                valores = por_pregunta.loc[configuracion, medida].reindex(consultas).to_numpy()
                filas.append({"medida": medida, "referencia": referencia, "configuracion": configuracion, "preguntas": len(consultas),
                              "media": valores.mean(), "diferencia": (valores - de_referencia).mean()})
    return pd.DataFrame(filas), consultas


def tabla_resumen(comparacion, medida):
    "media y diferencia contra cada referencia en una fila por configuracion"
    de_medida = comparacion[comparacion["medida"] == medida]
    tabla = de_medida[de_medida["referencia"] == REFERENCIAS[0]].set_index("configuracion")[["media"]]
    for referencia in REFERENCIAS:
        de_referencia = de_medida[de_medida["referencia"] == referencia].set_index("configuracion")
        tabla[f"menos {referencia}"] = de_referencia.apply(
            lambda f: f"{f['diferencia']:+.2f}", axis=1)
    return tabla.reindex(CONFIGURACIONES).round(3)


def cobertura_del_oro(tarea, muestra, claves):
    """por pregunta y configuracion de seleccion, la fraccion de items del oro con algun nodo en el subgrafo (sin modelo).
    en factoid el oro tiene un item y la fraccion es 0 o 1; en list es items cubiertos / len(oro)"""
    nodos_de_subgrafo = {}
    filas = []
    for nombre, consulta in muestra.items():
        nodos_por_item = [{tuple(nodo) for nodo in nodos} for nodos in consulta["oro_nodos"]]
        fila = {"consulta": nombre, "items_oro": len(nodos_por_item)}
        for configuracion in SELECCION:
            clave = (claves[nombre], configuracion)
            if clave not in nodos_de_subgrafo:
                ruta = os.path.join(tarea, "indices", claves[nombre], configuracion, "nodos.parquet")
                tabla = pd.read_parquet(ruta, columns=["label", "id"])
                nodos_de_subgrafo[clave] = set(zip(tabla["label"], tabla["id"]))
            cubiertos = sum(bool(nodos & nodos_de_subgrafo[clave]) for nodos in nodos_por_item)
            fila[f"cubiertos_{configuracion}"] = cubiertos
            fila[configuracion] = cubiertos / len(nodos_por_item)
        filas.append(fila)
    return pd.DataFrame(filas).set_index("consulta")


def grupo_de(fraccion):
    "el grupo de cobertura de una fraccion: hasta 0.5 inclusive es hasta_la_mitad, por encima mas_de_la_mitad"
    return GRUPOS[0] if fraccion <= 0.5 else GRUPOS[1]


def por_grupo(resultado, cobertura, medida):
    """la medida por grupo de cobertura, con cada configuracion de seleccion estratificada por su propio subgrafo, y
    sin_grafo y grafo_entero sobre las mismas preguntas como referencia apareada. una fila por configuracion y grupo"""
    por_pregunta = resultado["medidas"].groupby(["configuracion", "consulta"])[medida].mean()
    filas = []
    for configuracion in SELECCION:
        grupos = cobertura.loc[resultado["consultas"], configuracion].map(grupo_de)
        for grupo in GRUPOS:
            preguntas = grupos[grupos == grupo].index.tolist()  
            if not preguntas:
                continue
            valores = por_pregunta.loc[configuracion].reindex(preguntas).to_numpy()
            fila = {"configuracion": configuracion, "grupo": grupo, "preguntas": len(preguntas), medida: valores.mean()}
            for referencia in REFERENCIAS:
                de_referencia = por_pregunta.loc[referencia].reindex(preguntas).to_numpy()
                fila[f"{medida}_{referencia}"] = de_referencia.mean()
                fila[f"menos_{referencia}"] = (valores - de_referencia).mean()
            filas.append(fila)
    return pd.DataFrame(filas)


def origen_de_la_respuesta(tarea, resultado):
    """por respuesta con grafo, si el map puntuo algun reporte (si ninguno tuvo puntaje, el reduce no recibio nada de los
    reportes y la respuesta es memoria por construccion), si algun lote del map fallo (queda sin tokens de salida) y cuantas
    marcas [LLM: verify] puso el modelo. los subgrafos vacios no tienen contexto y no entran"""
    medidas = resultado["medidas"]
    filas = []
    for medida in medidas[medidas["configuracion"] != "sin_grafo"].itertuples():
        ruta = os.path.join(tarea, "respuestas", f"rep{medida.rep}", medida.consulta, medida.configuracion, "contexto_global_nivel0.json")
        if not os.path.exists(ruta):
            continue
        with open(ruta) as archivo:
            lotes = json.load(archivo)["map"]
        filas.append({"consulta": medida.consulta, "rep": medida.rep, "configuracion": medida.configuracion,
                      "map_con_puntos": any(punto["score"] > 0 for lote in lotes for punto in lote["respuesta"]),
                      "lote_fallado": any(lote["tokens_salida"] == 0 for lote in lotes),
                      "marcas": len(MARCA.findall(medida.respuesta))})
    return medidas.merge(pd.DataFrame(filas), on=["consulta", "rep", "configuracion"])


def tabla_origen(tarea, resultado):
    "por configuracion con grafo: cuantas respuestas tienen el map sin puntos, la medida principal segun el map y las marcas"
    origen = origen_de_la_respuesta(tarea, resultado)
    medida = MEDIDA_PRINCIPAL[tarea]
    grupos = origen.groupby("configuracion")
    tabla = pd.DataFrame({"respuestas": grupos.size(), "map_sin_puntos": 1 - grupos["map_con_puntos"].mean(),
                          "con_lote_fallado": grupos["lote_fallado"].mean(), "marcas_por_respuesta": grupos["marcas"].mean()})
    por_map = origen.groupby(["configuracion", "map_con_puntos"])[medida].mean().unstack("map_con_puntos")
    tabla[f"{medida}_map_sin_puntos"] = por_map.get(False)
    tabla[f"{medida}_map_con_puntos"] = por_map.get(True)
    con_marca = (origen["marcas"] > 0).groupby([origen["configuracion"], origen["map_con_puntos"]]).mean().unstack("map_con_puntos")
    tabla["con_marca_map_sin_puntos"] = con_marca.get(False)
    tabla["con_marca_map_con_puntos"] = con_marca.get(True)
    origen.drop(columns="respuesta").to_csv(f"origen_{tarea}.csv", index=False)
    return tabla.reindex(["grafo_entero"] + SELECCION).round(3)


def evaluar(tarea):
    muestra, claves, respuestas = cargar(tarea)
    print(f"{tarea}: {len(muestra)} preguntas en la muestra, {len(respuestas)} respuestas con codigo 0")
    if respuestas.empty:
        print(f"  sin respuestas todavia: falta correr responder.py --tarea {tarea}")
        return None
    display(pd.crosstab(respuestas["configuracion"], respuestas["rep"]).reindex(CONFIGURACIONES))
    medidas = pd.concat([respuestas.reset_index(drop=True), MEDIR[tarea](respuestas, muestra)], axis=1)
    medidas.drop(columns="respuesta").to_csv(f"medidas_{tarea}.csv", index=False)
    comparacion, consultas = comparar(medidas, COLUMNAS[tarea])
    if comparacion is None:
        print("  ninguna pregunta tiene las seis configuraciones")
    else:
        comparacion.to_csv(f"comparacion_{tarea}.csv", index=False)
        print(f"  {len(consultas)} preguntas con las seis configuraciones")
    return {"muestra": muestra, "claves": claves, "medidas": medidas, "comparacion": comparacion, "consultas": consultas}

## factoid: estricta, laxa y MRR

In [ ]:
factoid = evaluar("factoid")
if factoid:
    display(factoid["medidas"].groupby("configuracion")[COLUMNAS["factoid"] + ["abstiene", "segundos"]].mean().reindex(CONFIGURACIONES).round(3))
    print("por repeticion, para ver cuanto varia (laxa):")
    display(factoid["medidas"].pivot_table(index="configuracion", columns="rep", values="laxa", aggfunc="mean").reindex(CONFIGURACIONES).round(3))

In [ ]:
if factoid and factoid["comparacion"] is not None:
    print("media y diferencia apareada contra cada referencia, laxa:")
    display(tabla_resumen(factoid["comparacion"], "laxa"))

### el oro entra al subgrafo?



In [ ]:
if factoid and factoid["comparacion"] is not None:
    # con un solo item de oro la fraccion es 0 o 1: es el "algun nodo del oro adentro" de siempre
    adentro = cobertura_del_oro("factoid", factoid["muestra"], factoid["claves"])[SELECCION] > 0
    adentro.to_csv("cobertura_factoid.csv")
    print("preguntas con el oro adentro del subgrafo (A1):")
    display(pd.DataFrame({"preguntas": adentro.sum(), "fraccion": adentro.mean().round(3)}).reindex(SELECCION))
    medida = "laxa"
    por_pregunta = factoid["medidas"].groupby(["configuracion", "consulta"])[medida].mean()
    consultas = factoid["consultas"]
    filas = []
    for oro, grupo in [("adentro", [c for c in consultas if adentro.loc[c, "completa"]]), ("afuera", [c for c in consultas if not adentro.loc[c, "completa"]])]:
        if not grupo:
            continue
        referencia = por_pregunta.loc["sin_grafo"].reindex(grupo).to_numpy()
        for configuracion in CONFIGURACIONES:
            valores = por_pregunta.loc[configuracion].reindex(grupo).to_numpy()
            filas.append({"oro": oro, "preguntas": len(grupo), "configuracion": configuracion, medida: valores.mean(),
                          "menos_sin_grafo": (valores - referencia).mean()})
    a1 = pd.DataFrame(filas)
    a1.to_csv("a1_factoid.csv", index=False)
    display(a1.round(3))

## list: precisión, recall y F1

In [ ]:
list = evaluar("list")
if list:
    display(list["medidas"].groupby("configuracion")[COLUMNAS["list"] + ["abstiene", "segundos"]].mean().reindex(CONFIGURACIONES).round(3))
    print("por repeticion, para ver cuanto varia (f1):")
    display(list["medidas"].pivot_table(index="configuracion", columns="rep", values="f1", aggfunc="mean").reindex(CONFIGURACIONES).round(3))

In [ ]:
if list and list["comparacion"] is not None:
    print("media y diferencia apareada contra cada referencia, f1:")
    display(tabla_resumen(list["comparacion"], "f1"))

### cuanto del oro entra al subgrafo para list?



In [ ]:
if list and list["comparacion"] is not None:
    cobertura = cobertura_del_oro("list", list["muestra"], list["claves"])
    consultas = list["consultas"]
    print("items de oro por pregunta:")
    display(cobertura.loc[consultas, "items_oro"].describe(percentiles=[0.25, 0.5, 0.75]).round(2).to_frame().T)
    print("preguntas por largo de la lista de oro:")
    display(cobertura.loc[consultas, "items_oro"].value_counts().sort_index().to_frame().T)
    print("fraccion de items cubiertos, cuartiles por configuracion:")
    display(cobertura.loc[consultas, SELECCION].describe(percentiles=[0.25, 0.5, 0.75]).round(2))
    for configuracion in ["semillas", "completa"]:
        print(f"preguntas por largo de la lista (filas) e items cubiertos en {configuracion} (columnas):")
        display(pd.crosstab(cobertura.loc[consultas, "items_oro"], cobertura.loc[consultas, f"cubiertos_{configuracion}"]))

In [ ]:
if list and list["comparacion"] is not None:
    grupos = cobertura.loc[consultas, SELECCION].apply(lambda columna: columna.map(grupo_de))
    print("items del oro cubiertos, sobre el total de items:")
    cubiertos = cobertura.loc[consultas, [f"cubiertos_{c}" for c in SELECCION]].sum()
    cubiertos.index = SELECCION
    display(pd.DataFrame({"items": cubiertos, "fraccion": (cubiertos / cobertura.loc[consultas, "items_oro"].sum()).round(3)}))
    print("preguntas por grupo de cobertura, segun el subgrafo de cada configuracion:")
    display(pd.DataFrame({c: grupos[c].value_counts() for c in SELECCION}).reindex(GRUPOS).fillna(0).astype(int))
    for configuracion in SELECCION:
        cobertura[f"grupo_{configuracion}"] = grupos[configuracion]
    cobertura.to_csv("cobertura_list.csv")
    a1 = por_grupo(list, cobertura, "f1")
    a1.to_csv("a1_list.csv", index=False)
    print("f1 por configuracion y grupo (cada seleccion con su subgrafo), con las referencias sobre las mismas preguntas:")
    display(a1.round(3))

### list según cuánto del oro tiene nodo en el grafo entero



In [ ]:
#la misma tarea que list con su propia muestra y carpeta
MEDIR["list_por_oro"] = medir_list
COLUMNAS["list_por_oro"] = COLUMNAS["list"]
GRUPOS_ORO = ["nada", "menos_de_la_mitad", "la_mitad_o_mas"]

list_por_oro = evaluar("list_por_oro")
if list_por_oro:
    display(list_por_oro["medidas"].groupby("configuracion")[COLUMNAS["list"] + ["abstiene", "segundos"]].mean().reindex(CONFIGURACIONES).round(3))

In [ ]:
if list_por_oro and list_por_oro["comparacion"] is not None:
    muestra_por_oro = list_por_oro["muestra"]
    #f1 por pregunta y configuracion, media de las repeticiones
    por_pregunta = list_por_oro["medidas"].groupby(["configuracion", "consulta"])["f1"].mean()
    #fraccion de items del oro dentro del subgrafo de cada seleccion, por pregunta
    cobertura_por_oro = cobertura_del_oro("list_por_oro", muestra_por_oro, list_por_oro["claves"])
    filas = []
    for grupo in GRUPOS_ORO:
        #las preguntas del grupo que tienen las seis configuraciones
        preguntas_del_grupo = [c for c in list_por_oro["consultas"] if muestra_por_oro[c]["grupo_oro"] == grupo]
        if not preguntas_del_grupo:
            continue
        for configuracion in CONFIGURACIONES:
            valores = por_pregunta.loc[configuracion].reindex(preguntas_del_grupo).to_numpy()
            fila = {"grupo_oro": grupo, "preguntas": len(preguntas_del_grupo), "configuracion": configuracion, "f1": valores.mean()}
            #la diferencia apareada: cada pregunta contra si misma en la referencia
            for referencia in REFERENCIAS:
                de_referencia = por_pregunta.loc[referencia].reindex(preguntas_del_grupo).to_numpy()
                fila[f"menos_{referencia}"] = (valores - de_referencia).mean()
            #sin_grafo y grafo_entero no tienen subgrafo propio
            if configuracion in SELECCION:
                fila["oro_en_el_subgrafo"] = cobertura_por_oro.loc[preguntas_del_grupo, configuracion].mean()
            filas.append(fila)
    por_oro = pd.DataFrame(filas)
    por_oro.to_csv("por_oro_list.csv", index=False)
    display(por_oro.round(3))

### factoid sin nodo para el oro

La muestra de preguntas/armar_conjunto_sin_oro.ipynb: la pregunta entera en el grafo y el oro sin nodo. Las medidas son las de factoid, pero acá el diccionario no puede acertar y solo cuentan los sinónimos de BioASQ. Para comparar con factoid se usa laxa_sinonimos, que mide las dos muestras con la misma regla.


In [ ]:
#la misma tarea que factoid con su propia muestra y carpeta
MEDIR["factoid_sin_oro"] = medir_factoid
COLUMNAS["factoid_sin_oro"] = COLUMNAS["factoid"]

factoid_sin_oro = evaluar("factoid_sin_oro")
if factoid_sin_oro:
    display(factoid_sin_oro["medidas"].groupby("configuracion")[COLUMNAS["factoid"] + ["abstiene", "segundos"]].mean().reindex(CONFIGURACIONES).round(3))


## yes/no: exactitud y F1 macro

In [ ]:
yesno = evaluar("yesno")
if yesno:
    display(yesno["medidas"].groupby("configuracion")[COLUMNAS["yesno"] + ["abstiene", "segundos"]].mean().reindex(CONFIGURACIONES).round(3))
    print("por repeticion, para ver cuanto varia (acierto):")
    display(yesno["medidas"].pivot_table(index="configuracion", columns="rep", values="acierto", aggfunc="mean").reindex(CONFIGURACIONES).round(3))

In [ ]:
if yesno and yesno["comparacion"] is not None:
    # la F1 macro de cada configuracion sobre las preguntas de la comparacion (no es por pregunta: se calcula sobre el conjunto)
    de_la_comparacion = yesno["medidas"][yesno["medidas"]["consulta"].isin(yesno["consultas"])]
    filas = []
    for configuracion in CONFIGURACIONES:
        datos = de_la_comparacion[de_la_comparacion["configuracion"] == configuracion]
        filas.append({"configuracion": configuracion, "f1_macro": f1_macro(datos)})
    f1 = pd.DataFrame(filas).set_index("configuracion")
    f1.to_csv("f1_macro_yesno.csv")
    display(f1.round(3))

In [ ]:
if yesno and yesno["comparacion"] is not None:
    print("media y diferencia apareada contra cada referencia, acierto:")
    display(tabla_resumen(yesno["comparacion"], "acierto"))

## summary: ROUGE-2 y ROUGE-SU4

La respuesta ideal de BioASQ: un párrafo de hasta 200 palabras contra las respuestas ideales escritas por expertos, con la mejor de ellas por pregunta

In [ ]:
summary = evaluar("summary")
if summary:
    display(summary["medidas"].groupby("configuracion")[COLUMNAS["summary"] + ["palabras", "abstiene", "segundos"]].mean().reindex(CONFIGURACIONES).round(3))
    print("por repeticion, para ver cuanto varia (rouge_su4):")
    display(summary["medidas"].pivot_table(index="configuracion", columns="rep", values="rouge_su4", aggfunc="mean").reindex(CONFIGURACIONES).round(3))

In [ ]:
if summary and summary["comparacion"] is not None:
    print("media y diferencia apareada contra cada referencia, rouge_su4:")
    display(tabla_resumen(summary["comparacion"], "rouge_su4"))

### summary: cobertura y contradicción con el fact checker

summary se mide con RAGChecker (Ru et al., NeurIPS 2024), con gpt-oss:20b de juez: un modelo de otra familia
que el que respondió. El juez extrae las afirmaciones de la primera ideal (la del experto) una sola vez por pregunta y decide si
la respuesta entera implica, contradice o no dice cada una. Lo corre medir_summary_ragchecker.py; acá solo se
agrega.

- rc_cobertura: fracción de afirmaciones de la ideal que la respuesta implica (recall: lo que la respuesta agrega no resta)
- rc_contradichas, rc_con_contradiccion: afirmaciones de la ideal que la respuesta contradice, y si hay alguna.
- rc_afirmaciones_ideal: cuántas afirmaciones tiene la ideal (el denominador de la cobertura)


In [ ]:
if summary and summary["comparacion"] is not None:
    if summary["medidas"].loc[~summary["medidas"]["abstiene"], "rc_cobertura"].isna().all():
        print("falta correr el fact checker: medir_summary_ragchecker.py")
    else:
        print(f"respuestas medidas con el fact checker: {summary['medidas']['rc_cobertura'].notna().sum()} de {len(summary['medidas'])}")
        for medida in ["rc_cobertura", "rc_con_contradiccion", "rc_contradichas"]:
            print(f"media y diferencia apareada contra cada referencia, {medida}:")
            display(tabla_resumen(summary["comparacion"], medida))


### de dónde viene la respuesta? (yes/no y summary)



In [ ]:
for tarea, resultado in [("yesno", yesno), ("summary", summary)]:
    if resultado:
        print(f"{tarea}: fraccion de respuestas con el map sin puntos, {MEDIDA_PRINCIPAL[tarea]} segun el map y marcas")
        display(tabla_origen(tarea, resultado))

## Resumen



In [ ]:
RESULTADOS = {"factoid": factoid, "list": list, "yesno": yesno, "summary": summary}
filas = []
for tarea, resultado in RESULTADOS.items():
    if not resultado or resultado["comparacion"] is None:
        continue
    medida = MEDIDA_PRINCIPAL[tarea]
    tabla = tabla_resumen(resultado["comparacion"], medida)
    for configuracion, fila in tabla.iterrows():
        filas.append({"tarea": tarea, "medida": medida, "configuracion": configuracion, "media": fila["media"],
                      "menos grafo_entero": fila["menos grafo_entero"], "menos sin_grafo": fila["menos sin_grafo"]})
resumen = pd.DataFrame(filas)
resumen.to_csv("resumen.csv", index=False)
resumen

## Gasto por pregunta y etapa




Para contar cuantos tokens se gastan se ve en el log, que tiene un json de esta pinta: 

  ... - Metrics for openai/nomic-embed-text: {
    ...
  }
  ... - Metrics for openai/qwen3:30b-a3b-instruct-2507-q4_K_M: {
    "attempted_request_count": 7,
    "successful_response_count": 7,
    ...
    "prompt_tokens": 12228,   
    "completion_tokens": 2228, 
    "total_tokens": 14456,
    ...
  } 

  se filtra por "Metrics for openai/qwen3" para que se localice rapidamente el bloque del modelo que responde.



In [ ]:
GRAFO_ENTERO_TEXTO = "../grafo_entero/reportes250_map200_respuesta400_lotes5000_comunidades20_texto"
ETAPAS_TOKENS = ["tokens_deteccion", "tokens_indice", "tokens_consulta"]
ETAPAS_SEGUNDOS = ["segundos_deteccion", "segundos_enlace", "segundos_seleccion", "segundos_indice", "segundos_consulta"]


def uso_del_log(carpeta):
    """lamadas y tokens del modelo al escribir los reportes, sacado del bloque de metricas que graphrag deja al final de
    graphrag/logs/indexing-engine.log. es tokens_del_log de rag/graphrag_msft.py
    """
    ruta = os.path.join(carpeta, "graphrag", "logs", "indexing-engine.log")
    if not os.path.exists(ruta):
        return None
    with open(ruta) as archivo:
        texto = archivo.read()
    marca = "Metrics for openai/qwen3"
    if marca not in texto:
        return None
    bloque = texto.split(marca)[-1]
    metricas = json.loads(bloque[bloque.index("{"):bloque.index("\n}") + 2])
    return {"llamadas_indice": metricas["successful_response_count"], "tokens_entrada_indice": metricas["prompt_tokens"],
            "tokens_salida_indice": metricas["completion_tokens"]}


def gasto_consulta(tarea, medidas):
    "gasto requerido por cada consulta para cada respuesta, tanto en tokens como en segundos "
    filas = []
    for respuesta in medidas.itertuples():
        carpeta = os.path.join(tarea, "respuestas", f"rep{respuesta.rep}", respuesta.consulta, respuesta.configuracion)
        ruta = os.path.join(carpeta, "respuesta.json" if respuesta.configuracion == "sin_grafo" else "contexto_global_nivel0.json")
        uso = {"llamadas": 0, "tokens_entrada": 0, "tokens_salida": 0}
        if os.path.exists(ruta):
            with open(ruta) as archivo:
                uso = json.load(archivo)
        filas.append({"consulta": respuesta.consulta, "configuracion": respuesta.configuracion,
                      "llamadas_consulta": uso.get("llamadas", 1), "tokens_consulta": uso["tokens_entrada"] + uso["tokens_salida"],
                      "segundos_consulta": 0 if pd.isna(respuesta.segundos) else respuesta.segundos})
    return pd.DataFrame(filas).groupby(["consulta", "configuracion"], as_index=False)[["llamadas_consulta", "tokens_consulta", "segundos_consulta"]].mean()


def gasto_subgrafo(tarea, claves):
    "gasto en tiempo y en tokens de detectar las entidades"
    filas = []
    for clave in sorted(set(claves.values())):
        for configuracion in SELECCION:
            carpeta = os.path.join(tarea, "indices", clave, configuracion)
            with open(os.path.join(carpeta, "estadisticas.json")) as archivo:
                segundos_seleccion = json.load(archivo)["segundos_seleccion"]
            with open(os.path.join(carpeta, "indice.json")) as archivo:
                indice = json.load(archivo)
            fila = {"clave": clave, "configuracion": configuracion, "segundos_seleccion": segundos_seleccion,
                    "segundos_indice": 0, "llamadas_indice": 0, "tokens_indice": 0}
            if not indice.get("vacio"):
                if "tokens_entrada_indice" in indice:
                    uso = indice
                else:
                    uso = uso_del_log(os.path.join(tarea, indice["origen"]) if indice.get("reusado") else carpeta)
                fila["segundos_indice"] = indice["segundos"]
                fila["llamadas_indice"] = uso["llamadas_indice"] if uso else np.nan
                fila["tokens_indice"] = uso["tokens_entrada_indice"] + uso["tokens_salida_indice"] if uso else np.nan
            filas.append(fila)
    return pd.DataFrame(filas)


def gasto_enlace(tarea):
    "gasto en tiempo de enlazar los nodos"
    ruta = os.path.join("preguntas", f"enlace_{tarea}.json")
    if not os.path.exists(ruta):
        return None, None
    with open(ruta) as archivo:
        medido = json.load(archivo)
    enlace = pd.DataFrame.from_dict(medido["preguntas"], orient="index")
    enlace["tokens_deteccion"] = enlace["tokens_entrada"] + enlace["tokens_salida"]
    return enlace, medido["segundos_linkeador"]


def gasto_de(tarea, resultado):
    "llegar a un gasto en tokens y en segundos de todo el proceso"
    gasto = gasto_consulta(tarea, resultado["medidas"])
    gasto = gasto[gasto["consulta"].isin(resultado["consultas"])].copy()
    gasto["clave"] = gasto["consulta"].map(resultado["claves"])
    gasto = gasto.merge(gasto_subgrafo(tarea, resultado["claves"]), on=["clave", "configuracion"], how="left")
    enlace, segundos_linkeador = gasto_enlace(tarea)
    if enlace is None:
        for columna in ["tokens_deteccion", "segundos_deteccion", "segundos_enlace"]:
            gasto[columna] = np.nan
    else:
        gasto = gasto.merge(enlace[["tokens_deteccion", "segundos_deteccion", "segundos_enlace"]],
                            left_on="consulta", right_index=True, how="left")
        print(f"  deteccion y enlace medidos en {len(enlace)} preguntas, con las mismas entidades que la corrida en "
              f"{enlace['mismas_entidades'].mean():.0%}; armar el Linkeador: {segundos_linkeador} s, una vez por proceso")
    sin_subgrafo = ~gasto["configuracion"].isin(SELECCION)
    gasto.loc[sin_subgrafo, ["tokens_deteccion", "segundos_deteccion", "segundos_enlace", "segundos_seleccion",
                             "segundos_indice", "llamadas_indice", "tokens_indice"]] = 0
    faltan = {etapa: int(gasto[etapa].isna().sum()) for etapa in ETAPAS_TOKENS + ETAPAS_SEGUNDOS if gasto[etapa].isna().any()}
    if faltan:
        print(f"  ALERTA: sin medir (filas en NaN) {faltan}; los totales no las incluyen")
    gasto["tokens_total"] = gasto[ETAPAS_TOKENS].fillna(0).sum(axis=1)
    gasto["segundos_total"] = gasto[ETAPAS_SEGUNDOS].fillna(0).sum(axis=1)
    return gasto


def indice_estatico():
    "costo que tomo construir el indice del grafo entero (graphRAG estatico)"
    ruta = os.path.join(GRAFO_ENTERO_TEXTO, "indice.json")
    if not os.path.exists(ruta):
        return None
    with open(ruta) as archivo:
        segundos = json.load(archivo)["segundos_total"]
    uso = uso_del_log(GRAFO_ENTERO_TEXTO) or {"llamadas_indice": np.nan, "tokens_entrada_indice": np.nan, "tokens_salida_indice": 0}
    return {"segundos": segundos, "llamadas": uso["llamadas_indice"], "tokens": uso["tokens_entrada_indice"] + uso["tokens_salida_indice"]}

In [ ]:
estatico = indice_estatico()
#el gasto no necesita oro, asi que entran tambien los conjuntos que el resumen deja afuera
RESULTADOS_GASTO = dict(RESULTADOS)
RESULTADOS_GASTO["factoid_sin_oro"] = factoid_sin_oro
RESULTADOS_GASTO["list_por_oro"] = list_por_oro
MEDIDA_PRINCIPAL["factoid_sin_oro"] = "laxa"
MEDIDA_PRINCIPAL["list_por_oro"] = "f1"
for tarea, resultado in RESULTADOS_GASTO.items():
    if not resultado or resultado["comparacion"] is None:
        continue
    print(f"{tarea}: {len(resultado['consultas'])} preguntas")
    gasto = gasto_de(tarea, resultado)
    gasto.to_csv(f"gasto_por_pregunta_{tarea}.csv", index=False)
    columnas = ETAPAS_TOKENS + ["tokens_total", "llamadas_indice", "llamadas_consulta"] + ETAPAS_SEGUNDOS + ["segundos_total"]
    por_configuracion = gasto.groupby("configuracion")[columnas].mean().reindex(CONFIGURACIONES)
    por_configuracion["segundos_total_mediana"] = gasto.groupby("configuracion")["segundos_total"].median().reindex(CONFIGURACIONES)
    medida = MEDIDA_PRINCIPAL[tarea]
    de_la_comparacion = resultado["medidas"][resultado["medidas"]["consulta"].isin(resultado["consultas"])]
    por_configuracion[medida] = de_la_comparacion.groupby("configuracion")[medida].mean().reindex(CONFIGURACIONES)
    por_configuracion.to_csv(f"gasto_{tarea}.csv")
    print("minutos por pregunta y etapa, media (y la mediana del total):")
    minutos = (por_configuracion[ETAPAS_SEGUNDOS + ["segundos_total", "segundos_total_mediana"]] / 60).round(2)
    minutos.columns = [columna.replace("segundos_", "") for columna in minutos.columns]
    minutos[medida] = por_configuracion[medida].round(3)
    display(minutos)
    print("tokens y llamadas por pregunta y etapa:")
    display(por_configuracion[ETAPAS_TOKENS + ["tokens_total", "llamadas_indice", "llamadas_consulta"]].round(0))
    if estatico:
        ahorro = por_configuracion.loc["completa", "segundos_total"] - por_configuracion.loc["grafo_entero", "segundos_consulta"]
        print(f"indice de grafo_entero, pagado una vez y fuera de la tabla: {estatico['segundos'] / 3600:.1f} h, "
              f"{estatico['llamadas']:,.0f} llamadas, {estatico['tokens']:,.0f} tokens. completa gasta {ahorro:.0f} s mas por pregunta "
              f"que la consulta al grafo entero: el indice estatico se amortiza a las {estatico['segundos'] / ahorro:,.0f} preguntas.")
    else:
        print(f"falta {GRAFO_ENTERO_TEXTO}/indice.json: sin la linea del indice estatico")
    print()